# 05-02 Inspect Runnables: 체인 내부 들여다보기

앞 노트북에서 `RunnablePassthrough`로 체인을 구성해봤는데, 체인이 복잡해질수록 "지금 내 체인 구조가 맞는 건지" 확인하고 싶은 순간이 생겨요.

LCEL은 체인의 내부 구조를 **코드를 실행하지 않고도** 확인할 수 있는 검사(introspection) 도구를 제공해요. 이번 노트북에서는 이 도구들을 배워볼게요.

## 학습 목표

- `get_graph()`로 체인의 노드(Node)와 엣지(Edge) 구조를 확인하는 방법을 설명할 수 있어요
- `print_ascii()`로 체인 실행 흐름을 텍스트 다이어그램으로 시각화할 수 있어요
- `get_prompts()`로 체인 안에 포함된 프롬프트 내용을 확인하고 디버깅에 활용할 수 있어요

## 사전 지식

이 노트북을 시작하기 전에 다음 개념을 알고 있으면 좋아요:

- LCEL 파이프라인 구성 방법 (`|` 연산자)
- `RunnablePassthrough`와 `RunnableParallel`의 역할 (`01-RunnablePassthrough` 참고)
- RAG 체인의 기본 구조

---

> 🔑 **핵심 개념**: 체인 검사(Chain Introspection)란 **체인을 실행(invoke)하지 않고** 체인의 구조, 연결 관계, 프롬프트 내용을 확인하는 것을 말해요. 디버깅의 첫 번째 단계이기도 해요.

이 노트북에서 다루는 세 가지 검사 방법의 역할은 다음과 같아요:

| 검사 방법 | 확인할 수 있는 것 | 비유 |
|-----------|-------------------|------|
| `get_graph()` | 노드와 엣지 목록 | 지하철 노선의 역 목록과 연결 정보 |
| `print_ascii()` | 전체 구조 시각화 | 지하철 노선도 |
| `get_prompts()` | 프롬프트 내용과 변수 | 각 역에 붙어있는 안내문 |

```mermaid
flowchart LR
    C["LCEL 체인"]:::input
    G["get_graph()<br/>노드 + 엣지"]:::process
    P["print_ascii()<br/>ASCII 다이어그램"]:::process
    PR["get_prompts()<br/>프롬프트 목록"]:::process

    C --> G
    C --> P
    C --> PR

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
```

In [1]:
# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()


True

In [2]:
# ============================================================
# TODO: get_graph()로 검사할 RAG 체인을 구성하세요
# 힌트: {"context": retriever, "question": RunnablePassthrough()} | prompt | model | StrOutputParser()
# 예상 결과: "✅ RAG 체인 생성 완료" 출력
# ============================================================

from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# ---------------------------------------------------
# 검사 대상이 될 RAG 체인 구성하기
# ---------------------------------------------------

# 1단계: 예제 데이터로 FAISS 벡터 저장소(Vector Store) 생성
# - 이 데이터는 체인 검사용 예제이므로 내용보다 구조에 집중해요
vectorstore = FAISS.from_texts(
    ["파이썬은 데이터 과학과 머신러닝에 널리 사용되는 프로그래밍 언어입니다."],
    embedding=OpenAIEmbeddings(),
)

# 2단계: 벡터 저장소를 검색기(Retriever)로 변환
# - as_retriever(): 벡터 저장소를 Runnable 인터페이스를 가진 검색기로 변환
retriever = vectorstore.as_retriever()

# 3단계: RAG 프롬프트 템플릿 정의
# - {context}와 {question} 두 개의 변수를 사용
template = """다음 컨텍스트를 바탕으로 질문에 답변해주세요.
{context}  

질문: {question}"""

prompt = ChatPromptTemplate.from_template(template)

# 4단계: ChatOpenAI 모델 초기화
model = ChatOpenAI(model="gpt-4o-mini")

# 5단계: RAG 체인 생성
# - 이 체인의 내부 구조를 get_graph(), print_ascii(), get_prompts()로 검사할 예정
# - 딕셔너리 구문은 RunnableParallel로 자동 변환됨
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

print("=" * 60)
print("✅ RAG 체인 생성 완료")
print("=" * 60)

✅ RAG 체인 생성 완료


## 1. 그래프 구조 확인 - get_graph()

`chain.get_graph()` 메서드는 체인의 실행 그래프를 반환해요.

- **노드(Node)**: 체인의 각 단계(Retriever, Prompt, LLM, Parser 등)를 나타내요
- **엣지(Edge)**: 단계 간의 데이터 흐름 방향을 나타내요

이를 통해 체인의 전체 구조를 코드 없이도 파악할 수 있어요.

> 🔑 **핵심 개념**: `get_graph()`가 반환하는 그래프는 DAG(Directed Acyclic Graph, 유향 비순환 그래프)예요. 쉽게 말해, **데이터가 한 방향으로만 흐르고 순환(loop)이 없는** 구조를 의미해요. LCEL 체인은 항상 입력에서 출력으로 한 방향으로 진행되기 때문에 DAG 형태가 돼요.

> 🎯 **강의 포인트**: `get_graph()`는 디버깅 전용 도구예요. 학생들이 복잡한 체인을 직접 만든 뒤 이 메서드로 구조를 시각화해보면 LCEL 파이프라인을 훨씬 직관적으로 이해할 수 있어요. 특히 `RunnableParallel`을 사용한 병렬 분기가 그래프에서 어떻게 표현되는지 보면 이해가 빨라져요.

> ⚠️ **주의**: 노드 ID는 실행할 때마다 새로 생성되는 해시값이라 매번 달라져요. ID 자체보다 노드의 역할(소스와 타깃 관계)에 집중하세요.

In [3]:
# ============================================================
# TODO: chain.get_graph()를 호출하여 노드 목록을 출력하세요
# 힌트: graph = chain.get_graph(); nodes = graph.nodes
# 예상 결과: 노드 ID 해시값 8개 출력
# ============================================================

# ---------------------------------------------------
# 체인의 그래프에서 노드(Node) 확인하기
# ---------------------------------------------------

# 1단계: 그래프 객체 가져오기
# - get_graph(): 체인을 DAG(유향 비순환 그래프)로 표현한 Graph 객체 반환
# - 체인을 실행하지 않고도 구조를 분석할 수 있음
graph = chain.get_graph()

# 2단계: 노드 목록 추출
# - nodes: 체인의 각 처리 단계를 나타냄
# - 각 노드는 고유한 해시 ID를 가지며, Retriever, Prompt, Model, Parser 등에 해당
nodes = graph.nodes

print("=" * 60)
print("📊 체인의 노드 목록")
print("=" * 60)
for i, node in enumerate(nodes, 1):
    print(f"{i}. {node}")

📊 체인의 노드 목록
1. dbccdaa4459641078a2b67c0139d5625
2. 10a6eeae105b4e4994ad8af38ecff84b
3. 885c992db2bf4e2381f1e4ce4d37aa1f
4. 58c7773292b048bfbb64309ba0ffe308
5. 5cad5523c06c4cf1af75aea1e01b8210
6. 56d7fd777d6b4b9db182db127fa3b891
7. 515f015f910d4edca688ee9e74082ff1
8. 0fae93ecec7a480c94c6c691fa4d8fce


### 엣지(Edge)로 데이터 흐름 확인하기

노드 목록만으로는 "어떤 단계에서 어떤 단계로 데이터가 흐르는지" 알 수 없어요. 엣지(Edge)는 **노드 간 연결 방향**을 나타내요.

각 엣지는 `source`(출발 노드)와 `target`(도착 노드) 정보를 가지고 있어서, 데이터가 어디서 어디로 이동하는지 추적할 수 있어요.

> 💡 **실무 팁**: 엣지 목록에서 같은 `source`에서 여러 `target`으로 나가는 경우가 있다면, 그 지점이 `RunnableParallel`로 병렬 분기되는 곳이에요. 디버깅할 때 이 패턴을 눈여겨보세요.

In [4]:
# ---------------------------------------------------
# 체인의 그래프에서 엣지(Edge) 확인하기
# ---------------------------------------------------

# 엣지(Edge): 노드 간 데이터 흐름 방향
# - source: 데이터를 보내는 노드 (출발점)
# - target: 데이터를 받는 노드 (도착점)
# - conditional: 조건부 분기 여부 (False면 항상 실행)
edges = graph.edges

print("=" * 60)
print("🔗 체인의 엣지 목록 (데이터 흐름)")
print("=" * 60)
for i, edge in enumerate(edges, 1):
    print(f"{i}. {edge}")

🔗 체인의 엣지 목록 (데이터 흐름)
1. Edge(source='4f2a3b1799c24882ae2bac7e8af5ae32', target='b3095edb2ff146698313468b5bc652a7', data=None, conditional=False)
2. Edge(source='b3095edb2ff146698313468b5bc652a7', target='b444b0960bcd4d299201979a95f7b91c', data=None, conditional=False)
3. Edge(source='4f2a3b1799c24882ae2bac7e8af5ae32', target='8ed6ea43f0284c0594d28ebf736e32ab', data=None, conditional=False)
4. Edge(source='8ed6ea43f0284c0594d28ebf736e32ab', target='b444b0960bcd4d299201979a95f7b91c', data=None, conditional=False)
5. Edge(source='b444b0960bcd4d299201979a95f7b91c', target='df333098a0ef40e28d161fa42a5d88d0', data=None, conditional=False)
6. Edge(source='df333098a0ef40e28d161fa42a5d88d0', target='575c451970a44a169c35139e6124f693', data=None, conditional=False)
7. Edge(source='f5b512800a1a4e3383a8267bbcb3de2c', target='2b57e8a7c1d04da386519b610ebef082', data=None, conditional=False)
8. Edge(source='575c451970a44a169c35139e6124f693', target='f5b512800a1a4e3383a8267bbcb3de2c', data=None, condit

## 2. 그래프 시각화 - print_ascii()

노드 ID와 엣지 목록만으로는 체인의 전체 구조를 한눈에 파악하기 어려워요. `print_ascii()` 메서드를 사용하면 그래프를 **ASCII 아트 형식**으로 시각화할 수 있어요.

출력 결과를 보면 `Parallel<context,question>` 블록 안에서 `VectorStoreRetriever`와 `Passthrough`가 동시에 실행되고, 그 결과가 합쳐진 뒤 `ChatPromptTemplate → ChatOpenAI → StrOutputParser` 순서로 이어지는 구조를 한눈에 확인할 수 있어요.

> 🎯 **강의 포인트**: 체인이 예상과 다르게 동작할 때, 코드를 바로 수정하기보다 `print_ascii()`로 현재 구조를 먼저 확인하는 습관을 들이세요. "내가 의도한 구조"와 "실제 구조"가 다를 수 있기 때문이에요.

> 💡 **실무 팁**: 복잡한 RAG 파이프라인을 처음 디버깅할 때 `print_ascii()`를 먼저 실행해보면 예상치 못한 단계나 잘못 연결된 경로를 빠르게 찾을 수 있어요.

> ⚠️ **주의**: `print_ascii()`는 Jupyter 환경에서 가장 깔끔하게 출력돼요. 터미널 폭이 좁으면 출력이 깨질 수 있으니, 복잡한 체인은 넓은 터미널에서 확인하세요.

In [1]:
# ============================================================
# TODO: chain.get_graph().print_ascii()로 체인 구조를 시각화하세요
# 힌트: chain.get_graph().print_ascii()
# 예상 결과: Parallel<context,question> → ChatPromptTemplate → ChatOpenAI → StrOutputParser 흐름
# ============================================================

# ---------------------------------------------------
# 체인의 그래프를 ASCII 형식으로 시각화하기
# ---------------------------------------------------

# print_ascii(): 그래프를 터미널에서 읽을 수 있는 ASCII 다이어그램으로 출력
# - Parallel<...>: 병렬로 실행되는 구간을 표시
# - * 기호: 노드 간 연결(엣지)을 나타냄
# - 위에서 아래로 읽으면 데이터 흐름 순서를 파악할 수 있음
print("=" * 60)
print("📊 체인 구조 시각화 (ASCII)")
print("=" * 60)
chain.get_graph().print_ascii()

📊 체인 구조 시각화 (ASCII)


NameError: name 'chain' is not defined

## 3. 프롬프트 확인 - get_prompts()

체인에서 사용되는 프롬프트를 확인하는 것은 디버깅이나 최적화에 유용해요.

`chain.get_prompts()` 메서드는 체인에서 사용되는 **모든 프롬프트 객체의 리스트**를 반환해요. 프롬프트 템플릿의 내용뿐 아니라 어떤 변수(`input_variables`)가 필요한지도 한 번에 확인할 수 있어요.

> 🔑 **핵심 개념**: `get_prompts()`는 체인이 실행되기 전에 "이 체인에 어떤 프롬프트가 들어있는지"를 확인하는 도구예요. 멀티 체인 구조에서 중첩된 모든 서브체인의 프롬프트를 한꺼번에 돌려줘요.

> 💡 **실무 팁**: `input_variables`의 변수명이 `invoke()` 딕셔너리의 키와 정확히 일치하는지 검증하는 데 유용해요. 변수명 오타를 잡는 가장 빠른 방법이에요.

> ⚠️ **주의**: `input_variables`의 변수명과 `invoke()` 딕셔너리의 키가 다르면 `KeyError`가 발생해요. `get_prompts()`로 변수명을 먼저 확인하는 습관을 들이세요.

In [6]:
# ============================================================
# TODO: chain.get_prompts()로 체인에서 사용하는 프롬프트를 출력하세요
# 힌트: prompts = chain.get_prompts(); for p in prompts: print(p)
# 예상 결과: ChatPromptTemplate 1개, input_variables=['context', 'question']
# ============================================================

# ---------------------------------------------------
# 체인에서 사용되는 프롬프트 확인하기
# ---------------------------------------------------

# 1단계: get_prompts()로 체인 내 모든 프롬프트 객체 가져오기
# - 반환값: ChatPromptTemplate 객체의 리스트
# - 중첩된 서브체인 안의 프롬프트도 포함
prompts = chain.get_prompts()

print("=" * 60)
print("📝 체인에서 사용되는 프롬프트")
print("=" * 60)
print(f"프롬프트 개수: {len(prompts)}")
print()

# 2단계: 각 프롬프트의 타입과 내용 출력
# - type(): 어떤 종류의 프롬프트인지 확인 (ChatPromptTemplate 등)
# - input_variables: 이 프롬프트가 요구하는 변수명 목록
#   → invoke() 딕셔너리의 키와 일치해야 함
for i, prompt in enumerate(prompts, 1):
    print(f"프롬프트 {i}:")
    print(f"  타입: {type(prompt).__name__}")
    print(f"  내용: {prompt}")
    print()

📝 체인에서 사용되는 프롬프트
프롬프트 개수: 1

프롬프트 1:
  타입: ChatPromptTemplate
  내용: input_variables=['context', 'question'] input_types={} partial_variables={} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='다음 컨텍스트를 바탕으로 질문에 답변해주세요.\n{context}  \n\n질문: {question}'), additional_kwargs={})]



## 정리

### 체인 검사 메서드 비교

| 메서드 | 반환값 | 주요 용도 | 언제 사용할까? |
|--------|--------|-----------|---------------|
| `chain.get_graph().nodes` | 노드 딕셔너리 | 체인 구성 요소 목록 확인 | 체인에 어떤 단계가 포함됐는지 확인할 때 |
| `chain.get_graph().edges` | 엣지 리스트 | 데이터 흐름 방향 확인 | 병렬 분기나 연결 순서를 확인할 때 |
| `chain.get_graph().print_ascii()` | 없음 (출력만) | 전체 구조 시각화 | 체인이 예상대로 연결됐는지 한눈에 확인할 때 |
| `chain.get_prompts()` | 프롬프트 객체 리스트 | 프롬프트 내용 및 변수 검증 | `input_variables` 오타나 누락을 잡을 때 |

### 핵심 요점

- `get_graph()`는 체인의 실행 흐름을 DAG(유향 비순환 그래프) 형태로 돌려줘요
- `print_ascii()`는 병렬 실행 구조(`Parallel`)와 순차 실행 흐름을 한눈에 보여줘요
- `get_prompts()`는 프롬프트 변수와 내용을 빠르게 검증할 때 사용해요
- 노드 ID는 매 실행마다 달라지므로 구조 파악에만 활용해요

### 디버깅 워크플로 권장 순서

체인이 오류를 낼 때, 다음 순서로 확인하면 원인을 빠르게 찾을 수 있어요:

1. `print_ascii()`로 전체 구조가 의도한 대로인지 확인
2. `get_prompts()`로 `input_variables`와 `invoke()` 키가 일치하는지 확인
3. 문제가 특정 단계에 있다면 `get_graph().nodes`로 해당 노드를 식별

### 다음 노트북 예고

다음 노트북(`03-RunnableLambda-and-ChainDecorator.ipynb`)에서는 사용자가 직접 정의한 파이썬 함수를 LCEL 파이프라인에 연결하는 방법을 배워요. `RunnableLambda`와 `@chain` 데코레이터를 비교하면서 살펴볼게요.